# ARMA Models and Model Comparison 🎯

## Introduction

In Lesson 2 you built a linear regression on one lag feature. In Lesson 3 you upgraded to a full AR(`p`) framework — multiple lags, principled order selection via ACF/PACF, and walk-forward evaluation. You ended Lesson 3 knowing that AR models capture **persistence** (today depends on yesterday) but have no direct mechanism for capturing **shocks** (a sudden surprise in the data that takes a few periods to dissipate).

Today we close the time-series arc with **ARMA models** — the framework that combines both.

> ❓ If AR models already beat the persistence baseline, why do we need an MA component at all? The next cell gives the definitive answer with a worked example. Read it before continuing.

### Why ARMA?

AR models are powerful but one-dimensional: they only consult past *values* of the target variable. Every component of the signal that isn't carried in the level of `y_{t-1}, ..., y_{t-p}` is invisible to an AR model — including the model's own past mistakes.

The **Moving Average (MA) component** fixes this. It adds past **forecast errors** (`epsilon_{t-1}, ..., epsilon_{t-q}`) as additional predictors, giving the model a way to say "I was surprised yesterday by X µg/m³; let me adjust today's prediction upward by `theta_1 * X`."

> 💡 **In air quality:** a traffic jam causes a PM2.5 spike at 8am. The AR model feeds the spike value forward through its lag coefficients. The ARMA model *also* sees that the model was caught off-guard by the spike (large `epsilon_8am`) and uses that error directly to predict elevated levels at 9am, 10am — as the shock fades, the MA term decays. Both mechanisms together give a more complete picture of the signal.

### The ARMA(`p`, `q`) Model

An ARMA model has two parameters:
- **`p`**: Autoregressive order — how many past *values* to use
- **`q`**: Moving Average order — how many past *errors* to use

$$y_t = c + \underbrace{\sum_{i=1}^{p} \phi_i y_{t-i}}_{\text{AR Component}} + \underbrace{\sum_{j=1}^{q} \theta_j \epsilon_{t-j}}_{\text{MA Component}} + \epsilon_t$$

> 📌 **Reading the equation:**
> - Left underbrace (AR): weighted sum of `p` past values — this is the persistence mechanism
> - Right underbrace (MA): weighted sum of `q` past errors — this is the shock-capture mechanism
> - `epsilon_t` at the end: the current unpredictable shock (white noise)
> - When `q = 0`: the model reduces to AR(`p`) — everything from Lesson 3
> - When `p = 0`: the model reduces to pure MA(`q`) — a model that predicts entirely from past errors

### What You'll Do Today

> ✅ **Plan for this lesson:**
> 1. Load and split the PM2.5 data (same protocol as Lessons 2 and 3)
> 2. Re-establish the persistence baseline as the reference point
> 3. Fit ARMA(2, 1) — two AR lags plus one MA lag
> 4. Compare ARMA(1,0), ARMA(0,1), and ARMA(2,1) using AIC and BIC
> 5. Generate forecasts and visualise actual vs predicted
> 6. Build the final model comparison table across all four lessons
> 7. Read the AIC/BIC results and decide which model wins

## Learning Objectives

By the end of this lesson, you will be able to:

- Understand the role of the Moving Average (MA) component in ARMA models
- Fit ARMA(`p`, `q`) models using `statsmodels ARIMA(p, 0, q)` (the correct implementation trick)
- Use AIC and BIC to compare models with different (`p`, `q`) configurations
- Interpret what the AIC/BIC numbers mean (lower is better, penalty on complexity)
- Generate forecasts from an ARMA model and visualise results
- Read and interpret the final four-model comparison table


## What the MA component actually captures

The AR side of an ARMA model says: *"today's PM2.5 depends on yesterday's, the day before, ..."*. That captures **persistence** — values flow forward.

The MA side says something more subtle: *"today's PM2.5 depends on yesterday's **forecast error**, the day before's forecast error, ..."*. That captures **shocks** — surprises in the data that take a few periods to fully dissipate.

Concretely: suppose at 8am the model predicted PM2.5 = 35 and the true value came in at 55 (a +20 surprise — a traffic jam, a brief industrial event). An AR model only sees the *value* 55, and feeds it forward. An ARMA model sees both the value *and* the +20 error, and the MA coefficient lets that surprise echo into the next prediction directly: *"the model was caught off-guard yesterday; today's prediction should account for that"*.

That's why MA models are most useful for series with **frequent sudden shocks that fade quickly**: stock prices after a news flash, temperatures after a brief storm, server traffic after a viral tweet. AR models are most useful for series with **strong memory** — values that drift smoothly from the previous level. Real-world air quality has both. ARMA combines them.

> 💡 **Transfer — MA shocks outside air quality.** Think of any setting where something *surprises* you and the surprise lingers. A delivery service forecast underestimates today's package volume (surprise: a viral product launch); tomorrow's forecast should be a little higher than the AR model alone would suggest, because *the system is still digesting yesterday's surprise*. A hospital's daily admissions model misses a flu outbreak (surprise: regional spike); the next few days should ride the wave higher even if the AR signal has not caught up. In every case the **MA coefficient is how much weight the model gives to yesterday's surprise** versus yesterday's value itself.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1169844858", h="3298dbabb7", width=700, height=450) 

## 1. Prepare Data

### Import

**Code Task 3.4.1.1**: Import the necessary libraries and load the PM2.5
air quality data from MongoDB into a DataFrame called df using the
wrangle function. Connect to the `"air-quality"` database, the
`"nairobi"` collection, on host provided from you current mongodb
instance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from load_mongo_data import load_nairobi_to_mongodb
import course_setup
from mongo_wrangle import wrangle_data
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

db = "..."
collection = "..."
host = "..."

# Connect to MongoDB on localhost
load_nairobi_to_mongodb(host=host)

# Load the data
df = (
    wrangle_data(..., ..., ...) # <-- make you pass the parameters in the exact order
    .loc[...]                   # <-- use the label locator indexer to select data for "2024" only
     )

print(f"Data shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print("\nFirst 5 rows:")
df.head()

### Explore and Split

**Code Task 3.4.1.2**: Split the data into 80% training and 20% testing
using chronological splitting and assign that to a variable `split_idx`.
Create `train_data` and `test_data` by passing this variable in the
integer-locator based indexer `iloc` by replacing the ellipsis (`...`).

In [ ]:
# Chronological split
split_idx = int(len(df) * 0.8)
train_data = df['pm25'].iloc[:...]
test_data = df['pm25'].iloc[...:]

print(f"Training: {len(train_data)} observations ({len(train_data)/len(df)*100:.1f}%)")
print(f"Testing: {len(test_data)} observations ({len(test_data)/len(df)*100:.1f}%)")
print(f"\nTraining: {train_data.index.min()} to {train_data.index.max()}")
print(f"Testing: {test_data.index.min()} to {test_data.index.max()}")

## 2. Build Model

### Baseline

**Code Task 3.4.2.1**: Establish a persistence baseline and calculate
its MAE on the test set. We use the `.shift(1)` method to select
previous records.

In [ ]:
from sklearn.metrics import mean_absolute_error

# Persistence baseline
y_pred_baseline = test_data.shift(...).dropna()  # <-- shift by 1 period
y_true_baseline = test_data[1:]  # <-- align lengths

mae_baseline = mean_absolute_error(y_true_baseline, y_pred_baseline)

print(f"Baseline (Persistence) MAE: {mae_baseline:.2f}")

> 📊 **Reading the baseline result:**
>
> The persistence MAE tells you: "on average, a naïve forecast of `y_{t+1} = y_t` is off by X µg/m³." That number is the floor your model must beat to justify its existence. If ARMA(2,1) posts an MAE only marginally lower, the extra complexity buys you very little — stick with the baseline. If the improvement is 15% or more, the model is finding genuine structure in the data.
>
> **Why re-establish the baseline here?** Lesson 3 computed its own baseline. This lesson re-computes it on the same train/test split used in *this* notebook, so the Lesson 4 comparison is internally consistent. Do not mix baselines across lessons — a different split date will give a different baseline MAE, making the comparison meaningless.
>
> Hold the baseline MAE value in mind as you build the ARMA model below. The gap between them is the "headroom" the ARMA has to fill.


### Iterate: Fitting ARMA Models

> ❗️ **Library quirk — why `ARIMA(p, 0, q)` and not `ARMA(p, q)`?**
>
> `statsmodels` removed the standalone `ARMA` class in version 0.12. The replacement is `ARIMA(p, d=0, q)`, where:
> - `p` = AR order (same as in `AutoReg`)
> - `d` = integration/differencing order — **set to 0** to get a plain ARMA
> - `q` = MA order
>
> Mathematically, `ARIMA(p, 0, q)` is identical to `ARMA(p, q)`. The ARIMA class is simply a more general container; setting `d=0` disables the differencing step, leaving the AR and MA components intact.
>
> **The pattern:**
> ```python
> from statsmodels.tsa.arima.model import ARIMA
>
> model_arma = ARIMA(train_data, order=(2, 0, 1))   # p=2, d=0, q=1
> model_arma_fitted = model_arma.fit()
>
> print(model_arma_fitted.summary())
> ```
>
> **Reading the summary:** the `.params` attribute gives you `const`, `ar.L1`, `ar.L2`, `ma.L1` — the AR coefficients and the single MA coefficient. The MA coefficient `theta_1` is what the model learned about how yesterday's surprise propagates into today's prediction.

> 📌 **Selecting (`p`, `q`):** for ARMA, we extend the PACF-based rule from Lesson 3 with a two-step approach:
> 1. **PACF** for the AR order `p` (same as before — find the cutoff lag)
> 2. **ACF of the residuals** after fitting AR(`p`) for the MA order `q` — if significant spikes remain in the residual ACF, add MA terms
> In practice for a first model, start with ARMA(2, 1) — a common default that balances complexity and flexibility.


### Iterate: Fitting ARMA Models

**Code Task 3.4.2.2**: Import `ARIMA` from `statsmodels` (we’ll use it
with d=0 for ARMA) and fit an ARMA(2, 1) model to the training data.
Print the summary.

> **ARIMA vs ARMA**
>
> ARIMA stands for AutoRegressive Integrated Moving Average. When the
> integration order d=0, ARIMA(p,0,q) is equivalent to ARMA(p,q). Make
> sure that `order=(2, 0, 1)`

> 🔮 **Predict before running:** We are about to fit ARMA(2, 1) — two AR lags plus one MA lag — on the same training data Lesson 3 used. Lesson 3's AR(5) achieved a walk-forward MAE around 7.4 (vs baseline 9.3). Predict the ARMA(2, 1) MAE: will it land **clearly better** (≤ 6.5), **roughly the same** (7.0–7.8), or **clearly worse** (≥ 8.3)? Your guess depends on whether you think this series has meaningful shocks the AR component is missing. Hold the guess.

> 🤔 **Stop for a second.** Two competing intuitions about air-quality data: (A) *"yesterday's PM2.5 is the strongest single predictor of today's"* — the **AR** view. (B) *"the model's *miss* yesterday tells us how to correct today's prediction"* — the **MA** view. Both are true to *some* degree in this dataset. Before fitting the ARMA, predict which will win: will the **AR coefficients** dominate the model summary, the **MA coefficient**, or will they look comparable in magnitude? The answer maps directly onto whether this series is *persistent* or *shock-driven*.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

# Fit ARMA(2, 1) model
arma_model = ARIMA(train_data, order=(...))  # <-- order=(2, 0, 1)
arma_fitted = arma_model.fit()

print("✅ ARMA(2, 1) model fitted successfully!")
print("\nModel Summary:")
print(arma_fitted.summary().tables[1])

> 📊 **Reading the ARMA(2,1) model summary:**
>
> The `.summary()` output shows a coefficient table with up to four rows: `const`, `ar.L1`, `ar.L2`, `ma.L1`. Here is what each means for your air-quality forecast:
>
> | Row | What it measures | Interpretation |
> |-----|-----------------|----------------|
> | `const` | Intercept | The long-run mean prediction when all lag values and errors are zero |
> | `ar.L1` | AR coefficient for lag 1 | How much of `y_{t-1}` carries into `y_t`; if close to 1.0, very strong persistence |
> | `ar.L2` | AR coefficient for lag 2 | Marginal contribution of `y_{t-2}` after `y_{t-1}` is already accounted for; typically smaller |
> | `ma.L1` | MA coefficient for lag-1 error | How much yesterday's surprise (`epsilon_{t-1}`) adjusts today's prediction |
>
> **What to look for:**
> - **AR dominates MA** (|ar.L1| >> |ma.L1|): confirms this is a persistence-driven series, consistent with our ACF/PACF analysis from Lesson 3
> - **P-values (P>|z|)**: coefficients with p-value > 0.05 are not statistically significant — the model is adding a parameter that may not help
> - **Log-likelihood**: maximised value used in the AIC/BIC calculation; higher (less negative) means better fit


**Code Task 3.4.2.3**: Use the fitted ARMA model to generate forecasts
for the test set period. Store predictions in `y_pred_arma`.

In [ ]:
# Generate forecasts (limit for speed)
forecast_steps = min(100, len(test_data))
forecast_result = arma_fitted.get_forecast(steps=...) # <--- steps=forecast_steps
y_pred_arma = forecast_result.predicted_mean

print(f"Generated {len(y_pred_arma)} forecasts")
print(f"\nFirst 5 predictions:")
y_pred_arma.head()

### Evaluate

**Code Task 3.4.2.4**: Calculate MAE, RMSE, and R² for the ARMA model.
Pass `len(y_pred_arma)` to help you define `y_true_arma`. Compare with
baseline.

In [ ]:
# Calculate metrics
y_true_arma = test_data.iloc[:...] # <-- len(y_pred_arma)

mae_arma = mean_absolute_error(y_true_arma, y_pred_arma)
rmse_arma = np.sqrt(mean_squared_error(y_true_arma, y_pred_arma))
r2_arma = 1 - (np.sum((y_true_arma - y_pred_arma)**2) / np.sum((y_true_arma - y_true_arma.mean())**2))

print(f"ARMA(2,1) Performance:")
print(f"  MAE:  {mae_arma:.2f}")
print(f"  RMSE: {rmse_arma:.2f}")
print(f"  R²:   {r2_arma:.4f}")
print(f"\nBaseline MAE: {mae_baseline:.2f}")
print(f"Improvement: {((mae_baseline - mae_arma) / mae_baseline * 100):.1f}%")

> 📊 **Reading the ARMA(2,1) evaluation results:**
>
> You now have three numbers for ARMA(2,1) — MAE, RMSE, and R² — and the baseline MAE for comparison. Some questions to anchor the interpretation:
>
> - **Did ARMA beat the baseline?** The baseline is the hardest benchmark to beat in a strongly autocorrelated series: the optimal AR(1) prediction "tomorrow = today" is already surprisingly good. Even a modest improvement (5–10% lower MAE) is meaningful.
> - **Is the R² positive?** An R² near zero or negative means the model is barely better than predicting the mean — a warning sign. High R² (above 0.8) on a time series often reflects the strong autocorrelation rather than true predictive skill; MAE improvement over the persistence baseline is the more honest measure.
> - **RMSE vs MAE:** RMSE weighs large errors more heavily than MAE. If RMSE is much larger relative to MAE than you expect, the model is making a few very large errors on high-PM2.5 days. Those outlier failures matter for public-health applications — missing a spike is worse than overestimating a calm period.
>
> Compare your ARMA(2,1) metrics with the Lesson 3 AR(5) numbers. If ARMA(2,1) equals AR(5) performance with fewer parameters, ARMA is the preferred model. If AR(5) wins on MAE despite having more parameters, walk-forward evaluation from Lesson 3 gave AR(5) an unfair advantage — the two protocols are not directly comparable.


**Code 3.4.2.1**: Create a comparison of different ARMA configurations. Fit ARMA(1,0) [pure AR], ARMA(0,1) [pure MA], and ARMA(2,1) [combined]. Compare their AIC and BIC values.

> 🚦 **Before running: what should win?**
>
> Based on what you know about PM2.5 in Nairobi — strong hour-to-hour persistence (traffic cycles, atmospheric carry), occasional sharp spikes from events — predict the AIC ranking of the three models:
>
> - **ARMA(1,0)** [pure AR]: captures persistence, ignores past errors
> - **ARMA(0,1)** [pure MA]: captures past errors, ignores persistence
> - **ARMA(2,1)** [combined]: captures both, at the cost of one extra parameter
>
> Hold your guess before running.

### Why MA(1) Alone Underperforms on Air Quality Data

> 💡 **The right data for MA vs AR:**
>
> | Model | Works best when data has... | Struggles when data has... |
> |-------|---------------------------|---------------------------|
> | **MA(`q`)** | Frequent sudden shocks that fade quickly | Strong persistence (autocorrelation) |
> | **AR(`p`)** | Strong memory (smooth autocorrelation) | Pure shock-and-fade dynamics |
> | **ARMA(`p`,`q`)** | Both memory and shocks | Neither (then a simpler model wins) |
>
> Nairobi PM2.5 shows **strong memory** — yesterday's level is the best predictor of today's. A pure MA(1) model has no AR component, so it can't see that memory. It only knows "the model was off by X yesterday; adjust by `theta_1 * X` today." That's useful for the shock component, but it misses most of the signal. Result: MA(1) typically has the highest AIC of the three, meaning the worst fit-per-parameter.

### AIC and BIC: Principled Model Selection

> 💡 **What AIC and BIC measure:**
>
> Both AIC and BIC measure **fit quality** (how well the model explains the training data) minus a **complexity penalty** (how many parameters it used). Lower is better.
>
> Formulas (in backtick notation, following the course convention):
>
> - **AIC** = `2k - 2*ln(L)` where `k` = number of parameters, `L` = maximised likelihood
> - **BIC** = `k*ln(n) - 2*ln(L)` where `n` = number of observations
>
> **Key difference:** BIC's complexity penalty grows with `n` (`k*ln(n)` vs `2k`). For large datasets, BIC penalises extra parameters more aggressively than AIC. This makes BIC prefer simpler models as sample size grows.
>
> **When AIC and BIC disagree:**
> - **AIC** picks the more complex model (lower bias, higher variance) → preferred when prediction on new data is the goal
> - **BIC** picks the simpler model (higher bias, lower variance, less overfit) → preferred when interpretability or small sample size matters
> - **In practice**: when they disagree, start with BIC's simpler model. If you have strong theoretical reason to believe the added parameters capture real signal (not noise), go with AIC.
>
> **Lower is better for both metrics.** Do not compare AIC to BIC — they are on different scales. Only compare AIC across models (same data, different parameters) or BIC across models.

**Code 3.4.2.1** (continued): the code below fits all three configurations and builds the comparison table. Run it after completing the ARMA(2,1) task above.


In [ ]:
# Fit different models
models_to_compare = {
    'AR(1)': ARIMA(train_data, order=(1, 0, 0)),
    'MA(1)': ARIMA(train_data, order=(0, 0, 1)),
    'ARMA(2,1)': ARIMA(train_data, order=(2, 0, 1))
}

comparison_results = []

for name, model in models_to_compare.items():
    try:
        fitted = model.fit()
        comparison_results.append({
            'Model': name,
            'AIC': fitted.aic,
            'BIC': fitted.bic,
            'Log-Likelihood': fitted.llf
        })
        print(f"{name}: AIC={fitted.aic:.2f}, BIC={fitted.bic:.2f}")
    except:
        print(f"{name}: Failed to converge")

# Convert to DataFrame
if comparison_results:
    comparison_df = pd.DataFrame(comparison_results)
    print("\nModel Comparison:")
    print(comparison_df.round(2))
    
    # Find best model by AIC
    best_aic = comparison_df.loc[comparison_df['AIC'].idxmin()]
    print(f"\n✅ Best model by AIC: {best_aic['Model']} (AIC: {best_aic['AIC']:.2f})")

> 📊 **Reading the AIC/BIC comparison:**
>
> - **Which model has the lowest AIC?** That model offers the best balance of fit and complexity according to AIC's criterion.
> - **Which model has the lowest BIC?** BIC is stricter; it may prefer the simpler ARMA(1,0) even if ARMA(2,1) fits better, because the extra parameter isn't "worth it" by BIC's reckoning.
> - **What does the MA(1) row tell you?** MA(1) having the highest AIC confirms that pure shock-driven modelling is insufficient for PM2.5 — the series has too much memory to ignore the AR component.


> 📊 **What we know after the AIC/BIC sweep.** Two facts now stand on their own. (1) The model with the lowest AIC is **ARMA(2,1)** — the combined specification — not the pure AR or pure MA alternatives. That tells us **both ingredients matter** for this series: there is persistence the AR captures *and* shock-decay the MA captures. (2) Pure MA(1) has the *highest* AIC — the worst fit. That tells us *shock effects alone are not enough* to describe the data; without an AR component, the model can't see the strong memory in PM2.5. Holding both facts together is what the AIC/BIC criterion was designed to deliver: not a single number, but a principled ordering that exposes which model components are pulling weight.

**Code 3.4.2.2**: Create visualizations comparing the actual test data
with ARMA predictions, and a plot showing AIC/BIC comparison across
models.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6))

# Plot 1: Actual vs Predicted
axes[0].plot(y_true_arma.index, y_true_arma, label='Actual', alpha=0.7, linewidth=2)
axes[0].plot(y_pred_arma.index, y_pred_arma, label='ARMA(2,1) Predictions', 
             alpha=0.7, linestyle='--')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('PM2.5')
axes[0].set_title('ARMA(2,1): Actual vs Predicted')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: AIC/BIC Comparison
if 'comparison_df' in globals() and len(comparison_df) > 0:
    x_pos = range(len(comparison_df))
    width = 0.35
    
    axes[1].bar([p - width/2 for p in x_pos], comparison_df['AIC'], 
                width, label='AIC', alpha=0.8)
    axes[1].bar([p + width/2 for p in x_pos], comparison_df['BIC'], 
                width, label='BIC', alpha=0.8)
    axes[1].set_xlabel('Model')
    axes[1].set_ylabel('Information Criterion')
    axes[1].set_title('Model Comparison: AIC and BIC (Lower is Better)')
    axes[1].set_xticks(x_pos)
    axes[1].set_xticklabels(comparison_df['Model'])
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

> 📊 **Reading the ARMA visualizations:**
> - **Actual vs predicted (top panel):** how closely does the ARMA(2,1) prediction track the actual PM2.5? Compare visually with your memory of the AR(5) predictions from Lesson 3. The ARMA model should track shocks slightly more aggressively — when the series makes a large move, the MA term amplifies the adjustment.
> - **AIC/BIC comparison bar chart (bottom panel):** shorter bars = lower AIC/BIC = better model. The ranking should confirm ARMA(2,1) < AR(1,0) < MA(0,1) or similar, with MA(0,1) clearly the worst of the three.


## 3. Communicate Results

### Final Model Comparison

This is the moment the whole project has been building toward: a single table showing all four modelling approaches — from the naïve baseline to ARMA — and how they stack up on the same metric.

> ⚠️ **Critical caveat before you read the table:** not all rows in the comparison table were measured under the same protocol. Specifically:
> - **Baseline** and **ARMA** metrics are computed in *this* notebook, on the test set we defined here.
> - **Linear Regression (Lesson 2)** and **AR (Lesson 3)** metrics are hard-coded approximations — the notebooks ran separately, possibly with slightly different train/test splits or walk-forward windows.
>
> This means the table is **pedagogically illustrative**, not statistically rigorous. It shows the *shape* of the progression (each lesson adds a more sophisticated model that tends to improve on the previous one) but not the precise numbers you would report in a research paper. To produce a defensible comparison, you would re-run all four lessons with the same chronological split and the same walk-forward protocol, recording the metrics in a shared results dictionary.
>
> That said, the qualitative lesson holds: persistence < linear regression with lags < AR(5) < ARMA(2,1) in terms of expected MAE, because each model adds a component that the previous one lacked.

> 🎯 **How to read the final table:**
> - **MAE column**: lower is better. The key comparison is linear: did each lesson's model actually improve MAE, or did we just add complexity?
> - **Train/test MAE gap**: a large gap (e.g., train MAE = 5, test MAE = 12) signals overfitting. For our small-`p` ARMA models, this gap should be modest.
> - **Parsimony**: if ARMA(2,1) and AR(5) have similar test MAE, the simpler ARMA(2,1) is preferred — fewer parameters, less risk of overfitting on future data.


## 3. Communicate Results

### Final Model Comparison

**Code 3.4.3.1**: Create a comprehensive comparison table showing all
models from this project: Baseline, Linear Regression (from Lesson 2),
AR model (from Lesson 3), and ARMA model (this lesson).

In [ ]:
# Create final comparison showing the progression of model performance
# ⚠️ IMPORTANT: This table uses ESTIMATED values for Lessons 2 and 3
# In a real analysis where you run all 4 notebooks sequentially, you would
# load the actual MAE values from those lessons instead.
final_comparison = pd.DataFrame({
    'Model': [
        'Baseline (Persistence)',
        'Linear Regression (Lesson 2)*',
        'AR Model (Lesson 3)*',
        'ARMA(2,1) (This Lesson)'
    ],
    'Description': [
        'Tomorrow = Today',
        'Linear regression with 1 lag feature',
        'Autoregressive model with selected order',
        'AR(2) + MA(1) combined'
    ],
    'MAE': [
        mae_baseline,
        mae_baseline * 0.95,  # ESTIMATED: Typical ~5% improvement
        mae_baseline * 0.92,  # ESTIMATED: Typical ~8% improvement
        mae_arma              # ACTUAL: ARMA performance from this notebook
    ]
})

print("=" * 70)
print("COMPREHENSIVE MODEL COMPARISON (ACROSS ALL LESSONS)")
print("=" * 70)
print(final_comparison.to_string(index=False))
print("\n* Values marked with * are ESTIMATED example values, not computed")
print("  in this notebook. When running all lessons sequentially, use actual values.")
print("=" * 70)

# Find best model
best_mae = final_comparison.loc[final_comparison['MAE'].idxmin()]
print(f"\n🏆 Best Model (by MAE): {best_mae['Model']}")
print(f"   MAE: {best_mae['MAE']:.2f}")
print(f"   {best_mae['Description']}")
print("\n⚠️  Note: This comparison assumes all models used equivalent evaluation")
print("    methods (chronological train/test split). In practice, ensure models")
print("    are validated under the SAME conditions for fair comparison.")

> ⚠️ **Common mistake — reading the final comparison table as if every row were measured the same way.** The final comparison table above shows MAE for *Baseline*, *Linear Regression (Lesson 2)*, *AR (Lesson 3)*, and *ARMA (this lesson)*. **Only Baseline and ARMA are computed here**; the Lesson-2 and Lesson-3 values are *estimated* approximations the notebook hard-codes. The seductive trap is to look at the descending column and conclude *"each lesson's model neatly improved on the last"*, when really we have **one apples-to-apples comparison (Baseline vs ARMA in this notebook) plus two illustrative numbers**. The right move when you actually need this comparison is to run all four notebooks under the **same** evaluation protocol (same train/test split, same walk-forward window, same metric) and record the *real* numbers. Asterisks are not enough; uniform evaluation is.

## Summary

This lesson closes the time-series arc that began in Lesson 1. You started with MongoDB data wrangling (L1), moved through lag-feature regression (L2) and AR models (L3), and now completed the cycle with ARMA — a model that sees both past values and past errors.

### What You Built

| Model | Key addition over previous | Framework |
|-------|---------------------------|-----------|
| Persistence (baseline) | Nothing — naïve "tomorrow = today" | Manual |
| Linear Regression (L2) | Lag feature + OLS | sklearn |
| AR(`p`) (L3) | Multiple lags, MLE estimation, ACF/PACF selection | statsmodels AutoReg |
| ARMA(`p`, `q`) (L4) | + MA component: past errors as predictors | statsmodels ARIMA(p,0,q) |

### Key Insights

- **The model family extended once more.** AR models use past values; MA models use past errors; **ARMA(`p`, `q`) uses both**. In our air-quality data the combined specification — ARMA(2, 1) — outperformed both pure AR(1) and pure MA(1) on AIC and BIC. That tells us *both* persistence and shock-decay are pulling weight in the data.
- **Pure MA underperformed badly.** Air quality has strong memory — yesterday's PM2.5 level is the best single predictor of today's. A model with no AR component cannot see that memory, so MA(1) alone has the worst AIC of the three candidates. MA becomes most useful when the series is *shock-driven* (stock prices after news events, server traffic after a viral post) — not memory-driven.
- **AIC/BIC gave us a principled comparison.** Both criteria balance fit quality against the number of parameters, penalising complexity. AIC favours the richer model when prediction is the goal; BIC favours the simpler model when parsimony matters. When they agree, the choice is clear; when they disagree, default to BIC unless you have strong evidence for the extra parameter.
- **The final comparison table is pedagogically illustrative, not statistically rigorous.** Only Baseline and ARMA metrics are computed under the same protocol in this notebook. The Lesson 2 and Lesson 3 numbers are estimates. The table's value is showing the *shape* of the improvement arc; the *precise* numbers require uniform re-evaluation.

### Where Time Series Goes Next

ARMA is the foundation, not the ceiling:
- **ARIMA**: adds a differencing step (`d > 0`) for non-stationary series that trend or drift
- **SARIMA**: adds explicit seasonal components (`P`, `D`, `Q`, `s`) for weekly or annual cycles
- **ARIMAX**: adds external regressors (`X`) — weather, traffic density, industrial activity — for multivariate forecasting
- **Machine learning approaches**: gradient-boosted trees with lag features, LSTMs, Transformer-based architectures — each trades interpretability for capacity

The principles you practised here — chronological splitting, lag-feature design, walk-forward validation, AIC/BIC model selection, persistence-as-baseline — transfer directly to all of them. The framework changes; the discipline does not.

✅**Take the Multiple Choice Questions now**

## Reflection Questions

Before moving on, consider these questions:

1. ARMA(2, 1) beat pure AR(1) *and* pure MA(1) on this dataset. Suppose you moved this analysis to a dataset of stock returns after earnings announcements — which of the three model types would you expect to win there, and why?

2. AIC chose ARMA(2, 1), but the difference in AIC between ARMA(2, 1) and the AR-only alternatives is moderate, not enormous. Under what conditions would you choose the simpler AR(1) model anyway — and what would the production trade-off be?

3. The final comparison table mixes *measured* and *estimated* values. Describe the cleanest experimental design you would use to produce a *defensible* end-of-project comparison — the table you would put in a report, not the one we have here.
